# Aula 26: Árvores KD (KD-Trees) — Processamento Espacial de Nuvens de Pontos LiDAR no Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, implementaremos uma **Árvore $k$-dimensional (KD-Tree)** de 2 dimensões ($2D$) para acelerar o processamento de leituras do sensor LiDAR do **Visão-AGV**.

### Contexto no Projeto Visão-AGV
Durante a navegação em galpões industriais, o LiDAR do AGV captura continuamente centenas de pontos de reflexão por segundo no espaço 2D $(x, y)$. Para detectar os obstáculos mais próximos e evitar colisões em tempo de execução, uma busca exaustiva (força bruta $O(N)$) consome ciclos de processamento críticos do controlador.

A **KD-Tree** organiza os pontos do espaço particionando recursivamente os eixos alternadamente (eixo $X$ no nível 0, eixo $Y$ no nível 1, $X$ no nível 2, e assim por diante), reduzindo a complexidade de busca do vizinho mais próximo (*Nearest Neighbor*) para $O(\log N)$.

---

In [ ]:
# Implementação do Nó da Árvore KD (2D) e Construção Recursiva
import math
import time
import random

class NoKD:
    """
    Representa um nó na Árvore KD contendo um ponto 2D (x, y).
    """
    def __init__(self, ponto, esquerda=None, direita=None):
        self.ponto = ponto        # Tupla (x, y)
        self.esquerda = esquerda  # Subárvore com valores menores no eixo atual
        self.direita = direita    # Subárvore com valores maiores no eixo atual

def construir_kd_tree(pontos, profundidade=0):
    """
    Constrói uma KD-Tree balanceada a partir de uma lista de pontos 2D.
    Alterna os eixos de corte: eixo 0 (X) e eixo 1 (Y).
    """
    if not pontos:
        return None

    eixo = profundidade % 2

    # Ordena os pontos de acordo com o eixo atual e escolhe a mediana para balanceamento
    pontos.sort(key=lambda p: p[eixo])
    mediana = len(pontos) // 2

    return NoKD(
        ponto=pontos[mediana],
        esquerda=construir_kd_tree(pontos[:mediana], profundidade + 1),
        direita=construir_kd_tree(pontos[mediana + 1:], profundidade + 1)
    )

print("Estrutura e construção da KD-Tree declaradas com sucesso!")

## 2. Algoritmo de Busca do Vizinho Mais Próximo (Nearest Neighbor Search)

A busca percorre a árvore até a folha mais provável e realiza o corte de ramos (*pruning*) se a distância do e-corte em relação ao melhor ponto atual for maior que a menor distância encontrada.

In [ ]:
def dist_euclidiana_sq(p1, p2):
    """Calcula o quadrado da distância euclidiana entre dois pontos 2D (evita raiz quadrada para velocidade)."""
    return (p1[0] - p2[0])**2 + (p1[1] - p2[1])**2

def buscar_vizinho_mais_proximo(raiz, ponto_alvo, profundidade=0, melhor=None):
    """
    Encontra o ponto da nuvem LiDAR mais próximo do ponto alvo do AGV.
    """
    if raiz is None:
        return melhor

    if melhor is None:
        melhor = (raiz.ponto, dist_euclidiana_sq(ponto_alvo, raiz.ponto))
    else:
        dist_atual = dist_euclidiana_sq(ponto_alvo, raiz.ponto)
        if dist_atual < melhor[1]:
            melhor = (raiz.ponto, dist_atual)

    eixo = profundidade % 2
    
    # Escolhe a subárvore primária e secundária com base no eixo atual
    se_proximo = raiz.esquerda if ponto_alvo[eixo] < raiz.ponto[eixo] else raiz.direita
    se_distante = raiz.direita if ponto_alvo[eixo] < raiz.ponto[eixo] else raiz.esquerda

    # Desce pela subárvore mais promissora
    melhor = buscar_vizinho_mais_proximo(se_proximo, ponto_alvo, profundidade + 1, melhor)

    # Checa se vale a pena explorar a outra subárvore (Poda por hiperplano)
    if (ponto_alvo[eixo] - raiz.ponto[eixo])**2 < melhor[1]:
        melhor = buscar_vizinho_mais_proximo(se_distante, ponto_alvo, profundidade + 1, melhor)

    return melhor

print("Algoritmo de Busca por Vizinho Mais Próximo pronto!")

## 3. Simulação com Varredura LiDAR do Visão-AGV

Simularemos a geração de 1.000 pontos LiDAR do ambiente fabril e compararemos a velocidade de busca da KD-Tree contra a busca exaustiva (força bruta).

In [ ]:
random.seed(42)

# Simulação de 1000 pontos capturados pelo LiDAR no espaço 50m x 50m
nuvem_lidar = [(random.uniform(0, 50), random.uniform(0, 50)) for _ in range(1000)]
posicao_agv = (25.0, 25.0)  # Posição central do AGV

# Construção da Árvore
t_inicio_const = time.perf_counter()
raiz_kd = construir_kd_tree(list(nuvem_lidar))
t_fim_const = time.perf_counter()

# 1. Busca via KD-Tree
t_inicio_kd = time.perf_counter()
ponto_kd, dist_sq_kd = buscar_vizinho_mais_proximo(raiz_kd, posicao_agv)
t_fim_kd = time.perf_counter()

# 2. Busca Força Bruta (Linear)
t_inicio_fb = time.perf_counter()
ponto_fb = min(nuvem_lidar, key=lambda p: dist_euclidiana_sq(posicao_agv, p))
dist_sq_fb = dist_euclidiana_sq(posicao_agv, ponto_fb)
t_fim_fb = time.perf_counter()

print("=== COMPARATIVO DE DESEMPENHO NO PROCESSAMENTO LIDAR ===")
print(f"Pontos LiDAR processados : {len(nuvem_lidar)}")
print(f"Obstáculo mais próximo   : {ponto_kd} | Distância: {math.sqrt(dist_sq_kd):.2f} metros")
print(f"Tempo de Busca KD-Tree   : {(t_fim_kd - t_inicio_kd) * 1e6:.2f} µs")
print(f"Tempo de Busca Força Bruta: {(t_fim_fb - t_inicio_fb) * 1e6:.2f} µs")
print(f"Validação de Resultados  : {ponto_kd == ponto_fb}")

## 4. Análise de Eficiência e Conclusão

1. **Ganho de Desempenho:** A consulta via KD-Tree atinge complexidade de tempo médio de $O(\log N)$, permitindo processar varreduras LiDAR completas em microsegundos.
2. **Aplicação Prática no Visão-AGV:** Essa velocidade viabiliza checagens de colisão em tempo real no loop de controle do robô ($50\text{ Hz}$ ou mais).
3. **Conclusão da Etapa 03:** A KD-Tree encerra o módulo de Árvores Avançadas, fornecendo a base para o sistema de percepção e navegação do AGV.